# Notebook overview: R8_4-CO_explaining_centralities.ipynb

## What this notebook does
Analyzes and visualizes centrality and regression/model outputs for interpretation. It loads centrality/model result tables, extracts model coefficients, and creates publication-style plots that explain how network positions and covariates relate to outcomes.

## Inputs used
- Centrality CSVs generated by earlier graph notebooks
- Regression/model output CSVs
- Python helpers: extract_model_results.py and plot_regression_models.py

## Outputs created
- Cleaned model-result tables
- Coefficient/centrality plots
- Interpretive figures used for reporting network mechanisms

**Data access warning.** The Cuebiq/Spectus mobility stop data used by this project are proprietary/restricted and are not included in this repository. Do not commit raw mobility files, user IDs, stop tables, home-location files, graph outputs containing sensitive identifiers, or large derived files unless your data-use agreement explicitly permits release. Public users must obtain data access independently and place files in the documented paths.

In [ ]:
pip install statsmodels 

In [ ]:
pip install seaborn

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from sklearn.neighbors import BallTree
import numpy as np
from scipy.sparse import lil_matrix
import json
from collections import defaultdict
import networkx as nx
import pickle
import os
import matplotlib.pyplot as plt
from shapely.geometry import Point
from shapely import wkt
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score
import seaborn as sns

In [ ]:
base = os.path.join("..", "Data", "stop_df_perday_CO")
pois_dir = os.path.join(base, "POIs")
geo_dir = os.path.join(base, "geography_CO")
stops_dir = os.path.join(base, "daily_agg_to_weekly_Stops")
graph_poi_dir = os.path.join(base, "graphs", "poi-user")
os.makedirs(graph_poi_dir, exist_ok=True)
graph_dir = os.path.join(base, "graphs_POI_weighted")
os.makedirs(graph_dir, exist_ok=True)
home_dir = os.path.join(base, "home/pre_disaster")

In [ ]:
revision = "R11"

survivor_dir = os.path.join(graph_poi_dir, "boots_user_survivors", revision)
os.makedirs(survivor_dir, exist_ok=True)

In [ ]:
import sys
import os

sys.path.append(os.getcwd())

# Testing Hypothesis:

## 1
H1: Tie preservation is not random with respect to network position.
Nodes with higher structural importance in the pre-disaster network will be disproportionately preserved connectivityn relative to the random survival baseline.

Holding demographics and distance from disaster constant, individuals with higher pre-disaster centrality (strength, closeness) are more likely to remain active in the observed post-disaster network than expected under the random survival baseline.

Degree Strength (exposure proxy)
Hypothesis: Nodes with degree strength will show higher survival probability because they might have access to more resources and higher embededness.

Clustering coefficient (bonding proxy)
Hypothesis: Highly clustered nodes (embedded in tight triads) will show higher survival probability.(Coleman-style closure).

Closeness centrality (integration proxy)
Hypothesis: Nodes with higher closeness are more likely to remain active because they are embedded in well-integrated components.

Are high-centrality people the ones who remain post (vs random)? 

survival model :

POST: $Pr(S_i^{obs}=1)=\text{logit}^{-1}(\alpha + \gamma_1 C_i^{pre} + \gamma_2 H_i^{pre} + \gamma_3 Dist_i^{pre} + \gamma_4 Demog_i)$

RANDOM: $Pr(S_i^{rand})=\text{ols}^{-1}(\alpha + \delta_1 C_i^{pre} + \delta_2 H_i^{pre} + \delta_3 Dist_i^{pre} + \delta_4 Demog_i)$


H2, Who has higher centrality in pre? 
Individuals with higher third-place category entropy pre-disaster have higher pre-disaster degree centrality (especially strength; potentially closeness), because they participate in more distinct interaction contexts.
Individuals with more concentrated category profiles (lower entropy / stronger preference focus) have higher clustering (closure) because repeated co-presence occurs within fewer stable settings (more triadic closure)

$C_i^{pre} = \alpha + \beta_1 H_i^{pre} + \beta_2 Dist_i^{pre} + \beta_3 Demog_i + \epsilon_i$; $H_i$ - Entropy

$\beta_1 > 0$ for strength; clustering and closeness

## Individual level unique category level interactio- entropy, re-weighted by global baseline probabilities.
Using dyad level data (who visits larger variety of TP categories with someone else)

In [ ]:
dyad_path =  os.path.join(base, "for_ABM", f"individual_interac_allusers_{revision}_aligned.csv")
dyad = pd.read_csv(dyad_path)

In [ ]:
dyad.columns

In [ ]:
dyad["pre_sum"]  = pd.to_numeric(dyad["pre_sum"], errors="coerce").fillna(0)


In [ ]:
dyad = dyad.loc[dyad["pre_sum"] > 0].copy()

In [ ]:
# Symmetrize to ego-alter view ---
ego_i = dyad.rename(columns={"user_i": "user", "user_j": "alter"})
ego_j = dyad.rename(columns={"user_j": "user", "user_i": "alter"})
ego = pd.concat([ego_i, ego_j], ignore_index=True)

ego["user"] = ego["user"].astype(str)
ego["poi"] = ego["poi"].astype(str)
ego["SUB_CATEGORY"] = ego["SUB_CATEGORY"].astype(str)

In [ ]:
# user x category total pre interaction volume
user_cat = (
    ego.groupby(["user", "SUB_CATEGORY"], as_index=False)["pre_sum"].sum()
    .rename(columns={"pre_sum": "user_cat_pre"})
)

# user total pre
user_tot = user_cat.groupby("user", as_index=False)["user_cat_pre"].sum().rename(columns={"user_cat_pre": "user_total_pre"})
user_cat = user_cat.merge(user_tot, on="user", how="left")

# user's share of interactions in that category
user_cat["share_pre"] = np.where(
    user_cat["user_total_pre"] > 0,
    user_cat["user_cat_pre"] / user_cat["user_total_pre"],
    np.nan
)

In [ ]:
# global category probability (based on total pre interactions)
global_cat = (
    ego.groupby("SUB_CATEGORY", as_index=False)["pre_sum"].sum()
    .rename(columns={"pre_sum": "global_cat_pre"})
)
global_total = global_cat["global_cat_pre"].sum()
global_cat["prob_pre"] = np.where(global_total > 0, global_cat["global_cat_pre"] / global_total, np.nan)


In [ ]:
from scipy.stats import entropy

In [ ]:
# H_weighted_pre  (category-share weighted by global prob) - diversity across categories, but weighted toward globally common categories
# merge global prob onto user_cat
user_cat = user_cat.merge(global_cat[["SUB_CATEGORY", "prob_pre"]], on="SUB_CATEGORY", how="left")

# weight user's share by global popularity
user_cat["w_pre"] = user_cat["share_pre"] * user_cat["prob_pre"]

# normalize weights to a probability vector per user
den = user_cat.groupby("user")["w_pre"].transform("sum")
user_cat["w_pre_norm"] = np.where(den > 0, user_cat["w_pre"] / den, np.nan)

def entropy_safe(x):
    x = pd.to_numeric(x, errors="coerce")
    x = x[np.isfinite(x)]
    x = x[x > 0]
    return float(entropy(x)) if len(x) else np.nan

H_weighted_pre = (
    user_cat.groupby("user")["w_pre_norm"]
    .apply(entropy_safe)
    .rename("H_weighted_pre")
    .reset_index()
)

In [ ]:
# H_poi_pre  (POI-level entropy from user-POI shares - exploration / routine )
user_poi = (
    ego.groupby(["user", "poi"], as_index=False)["pre_sum"].sum()
    .rename(columns={"pre_sum": "user_poi_pre"})
)

user_poi_tot = user_poi.groupby("user", as_index=False)["user_poi_pre"].sum().rename(columns={"user_poi_pre": "user_total_pre"})
user_poi = user_poi.merge(user_poi_tot, on="user", how="left")

user_poi["p_poi_pre"] = np.where(
    user_poi["user_total_pre"] > 0,
    user_poi["user_poi_pre"] / user_poi["user_total_pre"],
    np.nan
)

H_poi_pre = (
    user_poi.groupby("user")["p_poi_pre"]
    .apply(entropy_safe)
    .rename("H_poi_pre")
    .reset_index()
)

In [ ]:
# Final user-level table (merge both)
user_entropy = H_weighted_pre.merge(H_poi_pre, on="user", how="outer")
user_entropy.head()

In [ ]:
user_entropy.shape

In [ ]:
unique_pois_pre = ego.groupby("user")["poi"].nunique().rename("unique_pois_pre").reset_index()
total_pre = ego.groupby("user")["pre_sum"].sum().rename("total_pre").reset_index()

poi_explore = unique_pois_pre.merge(total_pre, on="user", how="left")
poi_explore["poi_explore_rate_pre"] = np.where(poi_explore["total_pre"] > 0,
                                               poi_explore["unique_pois_pre"] / poi_explore["total_pre"],
                                               np.nan)

user_entropy = user_entropy.merge(poi_explore[["user","poi_explore_rate_pre"]], on="user", how="left")
user_entropy.head()

# Average distance traveled by each user to visit a third place

In [ ]:
dyad_nz = dyad.loc[(dyad["pre_sum"] > 0) | (dyad["post_sum"] > 0)].copy()

# --- parse geometries (WKT -> shapely) ---
# POI location: point_geometry
dyad_nz["poi_point"] = dyad_nz["point_geometry"].apply(lambda x: wkt.loads(x) if isinstance(x, str) else x)

# user i home centroid, user j home centroid
dyad_nz["home_i"] = dyad_nz["cbg_centroid_i"].apply(lambda x: wkt.loads(x) if isinstance(x, str) else x)
dyad_nz["home_j"] = dyad_nz["cbg_centroid_j"].apply(lambda x: wkt.loads(x) if isinstance(x, str) else x)

# --- build ego-view rows for i and j ---
ego_i = dyad_nz[["user_i","poi","poi_point","home_i"]].rename(
    columns={"user_i":"user","home_i":"home"}
)
ego_j = dyad_nz[["user_j","poi","poi_point","home_j"]].rename(
    columns={"user_j":"user","home_j":"home"}
)

ego = pd.concat([ego_i, ego_j], ignore_index=True)

# keep only rows with both home and poi geometry
ego = ego.dropna(subset=["user","poi_point","home"]).copy()
ego["user"] = ego["user"].astype(str)

# --- compute distances in meters using a projected CRS ---
gdf = gpd.GeoDataFrame(
    ego,
    geometry="poi_point",
    crs="EPSG:4326"
)
gdf["home_geom"] = gpd.GeoSeries(ego["home"], crs="EPSG:4326")

# project both to meters
gdf = gdf.to_crs(epsg=3857)
gdf["home_geom"] = gdf["home_geom"].to_crs(epsg=3857)

gdf["dist_m"] = gdf.geometry.distance(gdf["home_geom"])
gdf["dist_km"] = gdf["dist_m"] / 1000.0

# distance “to visit POIs where they had ≥1 interaction”
#  unique POIs, not repeated dyad rows -> dedupe (user, poi)
gdf_u = gdf.drop_duplicates(subset=["user","poi"]).copy()

# --- aggregate per user ---
user_avg_dist = (
    gdf_u.groupby("user", as_index=False)
    .agg(
        n_pois_interaction=("poi", "nunique"),
        avg_dist_km=("dist_km", "mean"),
        median_dist_km=("dist_km", "median"),
    )
)

user_avg_dist.head()

# Merge all user level information in centralities df

In [ ]:
csv_path = os.path.join(survivor_dir, f"combined_node_centralities_{revision}.csv")

In [ ]:
node_centralities = pd.read_csv(csv_path)
node_centralities.head()

In [ ]:
df = node_centralities.copy()
if "user_x" in df.columns:
    df["user"] = df["user_x"].fillna(df.get("node"))
elif "node" in df.columns:
    df["user"] = df["node"]
else:
    raise ValueError("No user identifier found (expected 'user_x' or 'node').")

df["user"] = df["user"].astype(str)
df["run"] = df["run"].astype(str)


In [ ]:
centrality_cols = [
    c for c in [
        "raw_degree",
        "strength",
        "closeness_centrality",
        "clustering_coefficient",
        "betweenness_centrality",   # include if exists
    ]
    if c in df.columns
]

In [ ]:
pre_mask = (df["phase"] == "pre")

pre_user = (
    df.loc[pre_mask, ["user"] + centrality_cols]
    .groupby("user", as_index=False)
    .mean(numeric_only=True)
    .rename(columns={c: f"{c}_pre_mean" for c in centrality_cols})
)

In [ ]:
post_mask = (df["phase"] == "post")

post_user = (
    df.loc[post_mask, ["user"] + centrality_cols]
    .groupby("user", as_index=False)
    .mean(numeric_only=True)
    .rename(columns={c: f"{c}_post_mean" for c in centrality_cols})
)

In [ ]:
rand_mask = (df["phase"] == "random_post")

random_user = (
    df.loc[rand_mask, ["user"] + centrality_cols]
    .groupby("user", as_index=False)
    .mean(numeric_only=True)
    .rename(columns={c: f"{c}_random_mean" for c in centrality_cols})
)

rand_n = (
    df.loc[rand_mask, ["user", "run"]]
    .drop_duplicates()
    .groupby("user")
    .size()
    .rename("n_random_runs_present")
    .reset_index()
)

random_user = random_user.merge(rand_n, on="user", how="left")


R = df.loc[rand_mask, "run"].astype(str).nunique()
random_user["n_random_runs_present"] = random_user["n_random_runs_present"].fillna(0).astype(int)
random_user["random_survival_prob"] = random_user["n_random_runs_present"] / R
random_user["random_evacuation_prob"] = 1 - random_user["random_survival_prob"]
random_user.head()

In [ ]:
user_centrality_table = (
    df.loc[pre_mask, ["user"]]
    .drop_duplicates()
    .merge(pre_user, on="user", how="left")
    .merge(post_user, on="user", how="left")
    .merge(random_user, on="user", how="left")
)

In [ ]:
user_centrality_table.head()

In [ ]:
user_entropy["user"] = user_entropy["user"].astype(str)

user_centrality_H = user_centrality_table.merge(
    user_entropy[["user", "H_weighted_pre", "H_poi_pre", "poi_explore_rate_pre"]],
    on="user",
    how="left"
)

In [ ]:
user_avg_dist["user"] = user_avg_dist["user"].astype(str)

user_centrality_H_dist = user_centrality_H.merge(
    user_avg_dist[["user", "n_pois_interaction", "avg_dist_km", "median_dist_km"]],
    on="user",
    how="left"
)

In [ ]:
user_centrality_H_dist.head()

In [ ]:
start_date = 20211001
end_date = 20211131
freq_home_path = os.path.join(home_dir, f"freq_home_{start_date}_{end_date}")
home_df = pd.read_csv(freq_home_path)
home_df = home_df.rename(columns={"cuebiq_id": "user", "fips_code": "pre_disaster_fips"})
home_df['geometry'] = home_df['geometry_wkt'].apply(wkt.loads)
home_df = gpd.GeoDataFrame(home_df, geometry='geometry', crs='EPSG:4326')
home_df["cbg_centroid"] = home_df.geometry.centroid
home_df.columns

In [ ]:
week_dir = os.path.join(graph_poi_dir, "user_evacuations")
output_path = os.path.join(week_dir, f"user_evacuations_{revision}.csv")
user_evac_df = pd.read_csv(output_path, low_memory=False)
user_evac_df.head()
# user_evac_df: columns ["user","evacuation"]
user_evac_df["user"] = user_evac_df["user"].astype(str)

# home_df: columns ["user","pre_disaster_fips"]
home_df = home_df.copy()
home_df["user"] = home_df["user"].astype(str)

ue = user_evac_df.merge(home_df[["user", "pre_disaster_fips", "cbg_centroid"]], on="user", how="left")

ue.head()

In [ ]:
ue.shape

In [ ]:
user_centrality_H_dist["user"] = user_centrality_H_dist["user"].astype(str)
ue["user"] = ue["user"].astype(str)
user_centrality_H_evac = user_centrality_H_dist.merge(
    ue[["user", "evacuation", "pre_disaster_fips", "cbg_centroid"]],
    on="user",
    how="left"
)
user_centrality_H_evac["evacuation"] = user_centrality_H_evac["evacuation"].fillna(-1).astype(int)
user_centrality_H_evac.head()

In [ ]:
user_centrality_H_evac.shape

In [ ]:
census_path = os.path.join(geo_dir, "colorado_cbg_census_only.csv")
census = pd.read_csv(census_path)
census["black_percent"] = census["black_alone"] / census["total_population"]
census["white_percent"] = census["white_alone"] / census["total_population"]
census.loc[census["median_income"] < 0, "median_income"] = 1
census.loc[census["median_rent"] < 0, "median_rent"] = 1
census["median_income_log"] = np.log1p(census["median_income"])
census["median_rent_log"] = np.log1p(census["median_rent"])
census["cbg_fips"] = census["cbg_fips"].astype(str)
census.head()

In [ ]:
user_centrality_H_evac["pre_disaster_fips"] = user_centrality_H_evac["pre_disaster_fips"].replace(r'\.0$', '', regex=True).astype(str)
user_centrality_H_evac.head()

In [ ]:
user_centrality_H_evac.shape

In [ ]:
user_centrality_H_evac_census = user_centrality_H_evac.merge(census,
    left_on="pre_disaster_fips", right_on="cbg_fips",
    how="left")
user_centrality_H_evac_census.head()

In [ ]:
# cbg_stats = os.path.join(base, "for_ABM/cbg_stats_census.csv")
# cbg_stats_census = pd.read_csv(cbg_stats)
# cbg_dist_fire = cbg_stats_census[["fips_code", "dist_epicentre"]].copy()
# cbg_dist_fire = cbg_dist_fire.rename(columns={"dist_epicentre": "fire_dist"})
# cbg_dist_fire.head()
from shapely.geometry import Point, shape
# ─── STEP 1: Paste your GeoJSON text here ───
geojson_text = """
{
  "type": "FeatureCollection",
  "name": "Marshall_Fire_Perimeter",
  "crs": {
    "type": "name",
    "properties": { "name": "EPSG:4326" }
  },
  "features": [
    {
      "type": "Feature",
      "properties": {},
      "geometry": {
        "type": "MultiPolygon",
        "coordinates": [
          [[
            [-105.199218, 39.979752],
            [-105.199394, 39.98012],
            [-105.200074, 39.980141],
            [-105.200157, 39.980633],
            [-105.199497, 39.980648],
            [-105.199501, 39.981102],
            [-105.198997, 39.981129],
            [-105.198975, 39.981415],
            [-105.19857, 39.9814],
            [-105.19857, 39.980965],
            [-105.198479, 39.980473],
            [-105.198822, 39.980473],
            [-105.199219, 39.980476],
            [-105.199218, 39.979752]
          ]]
        ]
      }
    }
  ]
}
"""

# ─── STEP 2: Convert the text to a dictionary ───
geojson_dict = json.loads(geojson_text)

# ─── STEP 3: Convert to list of geometries ───
geometries = [shape(feature["geometry"]) for feature in geojson_dict["features"]]

# ─── STEP 4: Create GeoDataFrame ───
gdf_fire = gpd.GeoDataFrame(
    {"event": ["Marshall Fire"] * len(geometries)},
    geometry=geometries,
    crs="EPSG:4326"
)

# ─── STEP 5: Preview ───
print(gdf_fire)

In [ ]:
gdf_fire = gdf_fire.to_crs(epsg=3857)

In [ ]:
user_centrality_H_evac_census = gpd.GeoDataFrame(user_centrality_H_evac_census, geometry="cbg_centroid", crs="EPSG:4326")
user_centrality_H_evac_census = user_centrality_H_evac_census.to_crs(epsg=3857)

In [ ]:
perimeter_centroid = gdf_fire.geometry.centroid.iloc[0]

In [ ]:
fire_geom = gdf_fire.geometry.unary_union

In [ ]:
user_centrality_H_evac_census["fire_dist"] = user_centrality_H_evac_census.geometry.distance(fire_geom)


In [ ]:
user_centrality_H_evac_census.shape

In [ ]:
#cbg_dist_fire["fips_code"] = cbg_dist_fire["fips_code"].astype(str)
#user_centrality_H_evac_census["pre_disaster_fips"] = user_centrality_H_evac_census["pre_disaster_fips"].astype(str)

In [ ]:
# user_centrality_H_evac_census_dist = user_centrality_H_evac_census.merge(cbg_dist_fire,
#     left_on="pre_disaster_fips", right_on="fips_code",
#     how="left")
# user_centrality_H_evac_census_dist.head()

In [ ]:
bins = pd.qcut(user_centrality_H_evac_census["fire_dist"], q=6, retbins=True, duplicates="drop")[1]

user_centrality_H_evac_census["fire_dist_bins"] = pd.cut(
    user_centrality_H_evac_census["fire_dist"],
    bins=bins,
    include_lowest=True
)

In [ ]:
user_centrality_H_evac_census.columns

In [ ]:
out_path = os.path.join(graph_dir, f"user_level_feature_table_{revision}.csv")
user_centrality_H_evac_census.to_csv(out_path, index=False)
print("Saved:", out_path)

# Regressions

In [ ]:
out_path = os.path.join(graph_dir, f"user_level_feature_table_{revision}.csv")
user_feature = pd.read_csv(out_path)

In [ ]:
user_feature.columns

In [ ]:
import importlib
import extract_model_results

In [ ]:
from extract_model_results import extract_model_results

In [ ]:
scale_cols = [
    "total_population", 
    "median_age",
    #"white_percent", 
    "black_percent",
    "median_income_log", 
    "median_rent_log", 
    "fire_dist",
    #"raw_degree_pre_mean",
    'strength_pre_mean',
    'closeness_centrality_pre_mean', 
    'clustering_coefficient_pre_mean',
    "H_weighted_pre",
   #"H_poi_pre", 
    "poi_explore_rate_pre", 
    "avg_dist_km",
    #"n_pois_interaction"
]

In [ ]:
sdm_cols = ["total_population","white_percent","median_income_log"]
dist_cols = ["fire_dist","avg_dist_km"]
mob_cols = ["H_weighted_pre","poi_explore_rate_pre"]
std_cols_pre = mob_cols + sdm_cols + dist_cols


scaler = StandardScaler()
user_feature[sdm_cols] = scaler.fit_transform(user_feature[sdm_cols])
user_feature[std_cols_pre] = scaler.fit_transform(user_feature[std_cols_pre])


In [ ]:
model_post = [
    ("Post_strength",
     "evacuation ~ strength_pre_mean + " + " + ".join(std_cols_pre)),
    
    ("Post_close",
     "evacuation ~ closeness_centrality_pre_mean + " + " + ".join(std_cols_pre)),
    
    ("Post_cluster",
     "evacuation ~ clustering_coefficient_pre_mean + " + " + ".join(std_cols_pre)),
]

model_random = [
    ("Random_strength",
     "random_evacuation_prob ~ strength_pre_mean + " + " + ".join(std_cols_pre)),
    
    ("Random_close",
     "random_evacuation_prob ~ closeness_centrality_pre_mean + " + " + ".join(std_cols_pre)),
    
    ("Random_cluster",
     "random_evacuation_prob ~ clustering_coefficient_pre_mean  + " + " + ".join(std_cols_pre)),
]

In [ ]:
all_results = []
fitted_models = {}

# -----------------------------
# Random baseline models (continuous probability -> OLS)
# -----------------------------
for model_name, formula in model_random:
    print(f"Running model: {model_name}")

    model_data = user_feature.copy()

    fit = smf.ols(formula=formula, data=model_data).fit()

    fitted_models[model_name] = fit
    res_df = extract_model_results(fit, model_name)
    all_results.append(res_df)

# -----------------------------
# Actual post evacuation models (binary -> Logit)
# -----------------------------
for model_name, formula in model_post:
    print(f"Running model: {model_name}")

    model_data = user_feature.copy()

    # optional but good practice: ensure binary and non-missing
    model_data = model_data[model_data["evacuation"].isin([0, 1])].copy()

    fit = smf.logit(formula=formula, data=model_data).fit(disp=False)

    fitted_models[model_name] = fit
    res_df = extract_model_results(fit, model_name)
    all_results.append(res_df)

results_df = pd.concat(all_results, ignore_index=True)


In [ ]:
for name, fit in fitted_models.items():
    print("\n" + "="*90)
    print(f"MODEL: {name}")
    print("="*90)
    print(fit.summary())

## Plot

In [ ]:
from plot_regression_models import plot_coef_ci

In [ ]:
var_pretty = {
    "strength_pre_mean": "Pre strength",
    "closeness_centrality_pre_mean": "Pre closeness",
    "clustering_coefficient_pre_mean": "Pre clustering",
    "fire_dist": "Dist from Fire",
    "total_population": "Population",
    "white_percent": "White (%)",
    "median_income_log": "Median income (log)",
}

model_pretty = {
    "Random_strength": "Random: Strength",
    "Random_close": "Random: Closeness",
    "Random_cluster": "Random: Clustering",
}

plot_coef_ci(
    results_df,
    models_to_plot=["Random_strength", "Random_close", "Random_cluster"],
    match_on="raw",
    var_order=[
        "strength_pre_mean",
        "closeness_centrality_pre_mean",
        "clustering_coefficient_pre_mean",
        "fire_dist",
        "total_population",
        "white_percent",
        "median_income_log",
    ],
    var_pretty=var_pretty,
    model_pretty=model_pretty,

    # ---- styling moved here ----
    model_colors={
        "Random: Strength": "#2ca02c",
        "Random: Closeness": "#1f77b4",
        "Random: Clustering": "#9467bd",
    },
    model_markers={
        "Random: Strength": "o",
        "Random: Closeness": "s",
        "Random: Clustering": "D",
    },
    marker_size=30,
    tick_fontsize=11,
    legend_fontsize=11,
    ci_lw=1.5,
    figsize=(7, 3),

    # ---- plot behavior ----
    drop_intercept=True,
    grey_non_sig=True,
    title="Random baseline evacuation probability",
    x_label="Coefficient",
)

In [ ]:
model_pretty_post = {
    "Post_strength": "Post: Strength",
    "Post_close": "Post: Closeness",
    "Post_cluster": "Post: Clustering",
}

plot_coef_ci(
    results_df,
    models_to_plot=["Post_strength", "Post_close", "Post_cluster"],
    match_on="raw",
    var_order=[
        "strength_pre_mean",
        "closeness_centrality_pre_mean",
        "clustering_coefficient_pre_mean",
        "H_weighted_pre",
        "total_population",
        "white_percent",
        "median_income_log",
    ],
    var_pretty=var_pretty,
    model_pretty=model_pretty_post,

    model_colors={
        "Post: Strength": "#2ca02c",
        "Post: Closeness": "#1f77b4",
        "Post: Clustering": "#9467bd",
    },
    model_markers={
        "Post: Strength": "o",
        "Post: Closeness": "s",
        "Post: Clustering": "D",
    },
    marker_size=30,
    tick_fontsize=11,
    legend_fontsize=11,
    ci_lw=1.5,
    figsize=(7, 3),

    drop_intercept=True,
    grey_non_sig=True,
    title="Observed post evacuation",
    x_label="Coefficient",
)